In [3]:
import numpy as np
import random
import pandas as pd
from scipy.stats import norm
import torch
import normflows as nf
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from itertools import combinations
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from scipy.interpolate import griddata
import seaborn as sns
from scipy.stats import norm
from tqdm import tqdm
import matplotlib.gridspec as gridspec

import os
import math

np.random.seed(0)

I = 5
tau = 0.2783047
full_df = pd.read_csv('case_by_case_output.csv')

# Cleaning

In [4]:
# Log transformation on the positive-valued variables
full_df['parallax'] = np.log(full_df['parallax'])
full_df['absorption'] = np.log(full_df['absorption'])
full_df['mass'] = np.log(full_df['mass'])

ids = full_df['ID'].unique()
cols = ['logAge', 'FeH', 'parallax', 'absorption', 'mass']

def CBC_sample_generator(num_samples=1, source_data=None):
    samples = source_data.sample(n=num_samples, replace=True).values
    return pd.DataFrame(samples, columns = cols)

#CBC_sample_generator(num_samples=5, source_data= full_df.drop(columns = ['file','ID']))

# Filtering extreme values and outliers
filtered_df = pd.DataFrame()
for i in ids:
    group = full_df[full_df["ID"] == i].copy()
    q = group["absorption"].quantile(0.007)
    group_filtered = group[group["absorption"] >= q]

    filtered_df = pd.concat([filtered_df, group_filtered], ignore_index=True)

full_df_original = full_df
full_df = filtered_df

# Standardize Data

In [5]:
data_means = []
data_stds = []

for id_val in ids:
    subset = full_df[full_df['ID'] == id_val]

    means = subset[cols].mean()
    stds = subset[cols].std()

    data_means.append(means.tolist())
    data_stds.append(stds.tolist())

data_normalized = full_df.copy()

data_normalized[cols] = (
    full_df
    .groupby('ID')[cols]
    .transform(lambda x: (x - x.mean()) / x.std())
)

In [6]:
data_normalized.to_csv('data_normalized.csv', index=False)

df_means = pd.DataFrame(data_means, index=ids, columns=cols)
df_stds = pd.DataFrame(data_stds, index=ids, columns=cols)

df_means.to_csv("df_means.csv")
df_stds.to_csv("df_stds.csv")

# Stage 1 Models

In [7]:
def train_stage1(id, data = data_normalized, latent_size = 5,
                 K = 3, hidden_units = 12, num_blocks = 2,
                 max_iter = 6000, num_samples = 300, lr = 1e-4, wd = 1e-6,
                 save_model_to = 'Forward_Models', save_loss_to = 'Loss'):
    
    globals().pop("nfm", None)
    globals().pop("loss_hist", None)
    torch.manual_seed(0)
    samples = data[data['ID'] == id].drop(columns=['ID', 'file'])

    flows = []
    for _ in range(K):
        flows += [nf.flows.AutoregressiveRationalQuadraticSpline(num_input_channels = latent_size,
                                                                    num_blocks = num_blocks, num_hidden_channels = hidden_units,
                                                                    init_identity=True)]
        flows += [nf.flows.LULinearPermute(latent_size)]


    q0 = nf.distributions.DiagGaussian(latent_size)
        
    nfm = nf.NormalizingFlow(q0=q0, flows=flows)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    nfm = nfm.to(device).float()

    optimizer = torch.optim.Adam(nfm.parameters(), lr=lr, weight_decay=wd)

    loss_hist = np.array([])

    for it in tqdm(range(max_iter)):
        optimizer.zero_grad()
        x = CBC_sample_generator(num_samples=num_samples, source_data=samples)
        x_tensor = torch.tensor(x.values, dtype=torch.float32).to(device)

        loss = nfm.forward_kld(x_tensor)
        
        if not (torch.isnan(loss) or torch.isinf(loss)):
            loss.backward()
            optimizer.step()
        
        loss_hist = np.append(loss_hist, loss.to('cpu').data.numpy())  

    torch.save(nfm, f"{save_model_to}/stage1_{id}.pth")
    torch.save(nfm.state_dict(), f"{save_model_to}/stage1_dict_{id}.pth")

    plt.figure(figsize=(6, 4))
    plt.plot(loss_hist, label='loss')
    plt.xlabel("Iteration")
    plt.ylabel("Forward KL Loss")
    plt.title(f"ID = {id}, Training Loss")
    plt.legend()
    plot_path = f"{save_model_to}/{save_loss_to}/ID{id}_loss_hist_stage1.png"
    plt.savefig(plot_path)
    plt.close()
    
    return nfm


In [8]:
for id in ids:
    model_name = f"stage1_id{id}"
    globals()[model_name] = train_stage1(id = id)

100%|██████████| 6000/6000 [00:40<00:00, 146.85it/s]


# Stage 1 Plots

In [9]:
color_map = {
        'logAge': '#B53717',
        'FeH': '#E99754',
        'parallax': '#2B9B8E',
        'absorption': '#0B3D51',
        'mass': '#FAD16A'
}
def plotting_function(model, id, latent_size=5, num_samples = 1000,
                      original_scale = True, save_to='Stage1_Figures', show=True, save=True):
    torch.manual_seed(0)
    np.random.seed(0)
    model.eval()

    # NF samples
    z, _ = model.sample(num_samples)
    z = z.detach().cpu().numpy()
    zz = pd.DataFrame(z, columns=cols)

    if original_scale:
        mu_list  = data_means[id-1]
        std_list = data_stds[id-1]
        mu_series  = pd.Series(mu_list, index=cols)
        std_series = pd.Series(std_list, index=cols)
        zz = zz * std_series + mu_series
        plotting_data = full_df[full_df['ID'] == id].drop(columns=['ID', 'file'])
    else:
        plotting_data = data_normalized[data_normalized['ID'] == id].drop(columns=['ID', 'file'])

    fig = plt.figure(figsize=(26, 14))
    gs = gridspec.GridSpec(
        2, 1, height_ratios=[0.7, 2.0], hspace=0.2, figure=fig
    )

    # -------- Row 1: Univariate densities --------
    uni_gs = gridspec.GridSpecFromSubplotSpec(
        1, latent_size, subplot_spec=gs[0], wspace=0.3
    )
    uni_axes = [fig.add_subplot(uni_gs[0, i]) for i in range(latent_size)]

    for i, name in enumerate(cols):
        color = color_map[name]
        sns.kdeplot(
            plotting_data[name], fill=True, color=color,
            label='True Density (KDE)', linewidth=2, alpha=0.6, ax=uni_axes[i]
        )
        sns.kdeplot(
            zz[name], color='black', label='NF-Estimated Density',
            linewidth=2, ax=uni_axes[i]
        )
        uni_axes[i].set_title(f"{name} Density (ID = {id})", fontsize=12)
        uni_axes[i].set_xlabel(name)


    # -------- Row 2: 2D densities --------
    twoD_gs = gridspec.GridSpecFromSubplotSpec(
        2, 5, subplot_spec=gs[1], wspace=0.25, hspace=0.35
    )
    ax_grid = [fig.add_subplot(twoD_gs[i, j]) for i in range(2) for j in range(5)]

    index = 0
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            var1, var2 = cols[i], cols[j]
            cmap = LinearSegmentedColormap.from_list(
                f"{var1}_{var2}_blend", [color_map[var1], color_map[var2]]
            )
            ax = ax_grid[index]
            sns.kdeplot(
                data=plotting_data, x=var1, y=var2,
                fill=True, cmap=cmap, thresh=0.1,
                levels=50, alpha=0.6, ax=ax
            )
            sns.kdeplot(
                data=zz, x=var1, y=var2,
                thresh = 0.1, levels = 10,
                fill=False, color='black', linewidths=1.2, ax=ax
            )
            ax.set_title(f"2D Density ({var1} vs {var2})", fontsize=11)
            ax.set_xlabel(var1)
            ax.set_ylabel(var2)
            index += 1

    if save:
        save_path = f'{save_to}/stage1_sampler_{id}.png'
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

    if show:
        plt.show()
    plt.close(fig)


In [10]:
for id in ids:
    model = globals()[f"stage1_id{id}"]
    plotting_function(model = model, id = id, num_samples = 5000,
                       original_scale= True,show = False, save = True, save_to='Stage1_Figures_OriginalScale')
    #plotting_function(model = model, id = id, data = data_normalized, num_samples = 5000,
    #                   original_scale= False,show = False, save = True, save_to='Stage1_Figures')